# Q-Shield XAI: Grad-CAM + SHAP Analysis on v3 Model

Generates paper figures showing WHICH QR regions trigger phishing detection.

**Inputs:**
- `siamese_v3_phase1.pth` (Siamese backbone)
- `classifier_v3_phase2.pth` (full classifier)
- Trad + CIC validation sets

**Outputs:**
- Grad-CAM heatmaps for 12 representative samples (benign vs phishing)
- SHAP analysis on classifier head embeddings
- Attention-weighted average heatmap per class

**Author:** Nicolas A. Llerena Silva (UTEC)

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
import sys, os, glob
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/QShield'
else:
    BASE = '.'

!pip install -q torch torchvision grad-cam shap matplotlib seaborn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
import pickle, zipfile, random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ============================================================
# 1. REBUILD MODEL ARCHITECTURE (same as v3)
# ============================================================

class MobileNetV2Embedding(nn.Module):
    def __init__(self, emb_dim=128, pretrained=False, dropout=0.35):
        super().__init__()
        mn = models.mobilenet_v2(weights=None)
        self.features = mn.features
        self.features[0][0] = nn.Conv2d(1, 32, 3, stride=2, padding=1, bias=False)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projection = nn.Sequential(
            nn.Linear(1280, 512), nn.BatchNorm1d(512), nn.ReLU(True),
            nn.Dropout(dropout), nn.Linear(512, emb_dim),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return F.normalize(self.projection(x), p=2, dim=1)


class QRClassifier(nn.Module):
    def __init__(self, backbone, emb_dim=128):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(emb_dim, 512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.BatchNorm1d(128), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(128, 32), nn.ReLU(True), nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


backbone = MobileNetV2Embedding(emb_dim=128, dropout=0.35)
classifier = QRClassifier(backbone, emb_dim=128).to(device)

# Load checkpoint
ckpt_path = os.path.join(BASE, 'classifier_v3_phase2.pth')
state = torch.load(ckpt_path, map_location=device)
classifier.load_state_dict(state)
classifier.eval()
print(f'Loaded v3 classifier from {ckpt_path}')
print(f'Params: {sum(p.numel() for p in classifier.parameters()):,}')

In [ ]:
# ============================================================
# 2. LOAD SAMPLE DATA (Trad for visualization)
# ============================================================
WORK = '/content/qshield_data'
trad_dir = os.path.join(WORK, 'trad')

if not os.path.exists(os.path.join(trad_dir, 'qr_codes_29.pickle')):
    os.makedirs(trad_dir, exist_ok=True)
    with zipfile.ZipFile(os.path.join(BASE, 'QuishingDataset.zip')) as z:
        z.extractall(trad_dir)

with open(os.path.join(trad_dir, 'qr_codes_29.pickle'), 'rb') as f:
    trad_qr = pickle.load(f)
with open(os.path.join(trad_dir, 'qr_codes_29_labels.pickle'), 'rb') as f:
    trad_labels = pickle.load(f)

print(f'Loaded Trad: {trad_qr.shape}')

---
## 3. GRAD-CAM — Visual Attention Maps

For each sample, we compute the gradient of the classification logit w.r.t. the last conv features,  
then weight the feature maps by these gradients to produce a heatmap showing WHICH regions  
of the QR code triggered the phishing prediction.

In [ ]:
# ============================================================
# 3.1 GRAD-CAM IMPLEMENTATION (pure PyTorch, no external lib needed)
# ============================================================

class GradCAM:
    """Grad-CAM for MobileNetV2. Hooks into last conv layer."""

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, input_tensor, target_class=None):
        self.model.eval()
        logit = self.model(input_tensor)
        if target_class is None:
            target_class = (torch.sigmoid(logit) > 0.5).long().squeeze()

        self.model.zero_grad()
        logit.backward(torch.ones_like(logit))

        # Weighted sum of activations
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # GAP over spatial dims
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        # Normalize to [0, 1]
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


# Target layer: last inverted residual block of MobileNetV2
target_layer = classifier.backbone.features[-1]
gradcam = GradCAM(classifier, target_layer)
print('Grad-CAM ready.')

In [ ]:
# ============================================================
# 3.2 GENERATE HEATMAPS FOR SAMPLES
# ============================================================

def prepare_tensor(qr_array):
    """Convert 69x69 binary array to 1x1x224x224 float tensor."""
    t = torch.from_numpy(qr_array.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=False)
    return t.to(device)

# Sample 6 benign and 6 phishing for visualization
np.random.seed(42)
benign_idx = np.where(trad_labels == 0)[0]
phish_idx = np.where(trad_labels == 1)[0]
benign_sample = np.random.choice(benign_idx, 6, replace=False)
phish_sample = np.random.choice(phish_idx, 6, replace=False)

# Generate heatmaps
fig, axes = plt.subplots(4, 6, figsize=(18, 12))

for col, (idx, true_lbl) in enumerate([(i, 0) for i in benign_sample]):
    qr = trad_qr[idx]
    tensor = prepare_tensor(qr)
    tensor.requires_grad_(True)
    with torch.enable_grad():
        cam = gradcam(tensor)
    with torch.no_grad():
        prob = torch.sigmoid(classifier(prepare_tensor(qr))).item()

    # Original
    axes[0, col].imshow(qr, cmap='gray')
    axes[0, col].set_title(f'Benign #{idx}\nPred: {prob:.3f}', fontsize=9,
                            color='green' if prob < 0.5 else 'red')
    axes[0, col].axis('off')
    # Heatmap overlay
    axes[1, col].imshow(qr, cmap='gray')
    im = axes[1, col].imshow(cam, cmap='jet', alpha=0.55)
    axes[1, col].axis('off')

for col, (idx, true_lbl) in enumerate([(i, 1) for i in phish_sample]):
    qr = trad_qr[idx]
    tensor = prepare_tensor(qr)
    tensor.requires_grad_(True)
    with torch.enable_grad():
        cam = gradcam(tensor)
    with torch.no_grad():
        prob = torch.sigmoid(classifier(prepare_tensor(qr))).item()

    axes[2, col].imshow(qr, cmap='gray')
    axes[2, col].set_title(f'Phishing #{idx}\nPred: {prob:.3f}', fontsize=9,
                            color='red' if prob > 0.5 else 'green')
    axes[2, col].axis('off')
    axes[3, col].imshow(qr, cmap='gray')
    axes[3, col].imshow(cam, cmap='jet', alpha=0.55)
    axes[3, col].axis('off')

# Row labels
axes[0, 0].set_ylabel('Benign\n(original)', fontsize=11, fontweight='bold', rotation=0, labelpad=50)
axes[1, 0].set_ylabel('Benign\n(Grad-CAM)', fontsize=11, fontweight='bold', rotation=0, labelpad=50)
axes[2, 0].set_ylabel('Phishing\n(original)', fontsize=11, fontweight='bold', rotation=0, labelpad=50)
axes[3, 0].set_ylabel('Phishing\n(Grad-CAM)', fontsize=11, fontweight='bold', rotation=0, labelpad=50)

fig.suptitle('Grad-CAM Attention Maps — Q-Shield v3 Classifier',
             fontweight='bold', fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_gradcam_samples.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_gradcam_samples.png')

In [ ]:
# ============================================================
# 3.3 AVERAGE ATTENTION MAP PER CLASS
# ============================================================
# Aggregate Grad-CAM across many samples to see WHERE the model consistently looks

N_SAMPLES = 200
np.random.seed(42)
b_aggregate_idx = np.random.choice(benign_idx, N_SAMPLES, replace=False)
p_aggregate_idx = np.random.choice(phish_idx, N_SAMPLES, replace=False)

b_cams = []
p_cams = []

print('Computing aggregate Grad-CAM (200 benign + 200 phishing)...')
for i in b_aggregate_idx:
    tensor = prepare_tensor(trad_qr[i])
    tensor.requires_grad_(True)
    with torch.enable_grad():
        cam = gradcam(tensor)
    b_cams.append(cam)

for i in p_aggregate_idx:
    tensor = prepare_tensor(trad_qr[i])
    tensor.requires_grad_(True)
    with torch.enable_grad():
        cam = gradcam(tensor)
    p_cams.append(cam)

avg_benign_cam = np.mean(b_cams, axis=0)
avg_phish_cam = np.mean(p_cams, axis=0)
diff_cam = avg_phish_cam - avg_benign_cam

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

im0 = axes[0].imshow(avg_benign_cam, cmap='jet', vmin=0, vmax=1)
axes[0].set_title('Average attention — Benign QRs', fontweight='bold')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], shrink=0.7)

im1 = axes[1].imshow(avg_phish_cam, cmap='jet', vmin=0, vmax=1)
axes[1].set_title('Average attention — Phishing QRs', fontweight='bold')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], shrink=0.7)

im2 = axes[2].imshow(diff_cam, cmap='RdBu_r', vmin=-abs(diff_cam).max(), vmax=abs(diff_cam).max())
axes[2].set_title('Difference (Phishing - Benign)', fontweight='bold')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], shrink=0.7)

fig.suptitle('Class-Level Spatial Attention Analysis (n=200 per class)',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_gradcam_aggregate.png'), dpi=300, bbox_inches='tight')
plt.show()

print('\nKey insight for paper:')
print('Where the model attends tells us WHAT structural regions differentiate phishing QRs.')

---
## 4. EMBEDDING ANALYSIS — Inter-class distances

In [ ]:
# ============================================================
# 4.1 DISTANCE DISTRIBUTION IN EMBEDDING SPACE
# ============================================================
from sklearn.model_selection import train_test_split

# Split to get val set
idx_tr, idx_val = train_test_split(np.arange(len(trad_labels)), test_size=0.2,
                                    stratify=trad_labels, random_state=42)
qr_val = trad_qr[idx_val]
lab_val = trad_labels[idx_val]

# Compute embeddings
classifier.eval()
embeddings = []
with torch.no_grad():
    for i in range(0, len(qr_val), 128):
        batch = qr_val[i:i+128]
        tensors = torch.stack([prepare_tensor(q).squeeze(0) for q in batch])
        embeddings.append(classifier.backbone(tensors).cpu().numpy())
embeddings = np.concatenate(embeddings)
print(f'Embeddings shape: {embeddings.shape}')

# Compute pairwise distances within/between classes
from scipy.spatial.distance import pdist, squareform

b_emb = embeddings[lab_val == 0]
p_emb = embeddings[lab_val == 1]

# Sample for speed
SAMPLE = 500
if len(b_emb) > SAMPLE: b_emb = b_emb[np.random.choice(len(b_emb), SAMPLE, replace=False)]
if len(p_emb) > SAMPLE: p_emb = p_emb[np.random.choice(len(p_emb), SAMPLE, replace=False)]

# Intra-class
bb_dist = pdist(b_emb)
pp_dist = pdist(p_emb)

# Inter-class
bp_dist = []
for i in range(min(200, len(b_emb))):
    for j in range(min(200, len(p_emb))):
        bp_dist.append(np.linalg.norm(b_emb[i] - p_emb[j]))
bp_dist = np.array(bp_dist)

fig, ax = plt.subplots(figsize=(10, 6))
bins = np.linspace(0, max(bb_dist.max(), pp_dist.max(), bp_dist.max()), 60)
ax.hist(bb_dist, bins=bins, alpha=0.5, color='#2ecc71', label=f'Benign-Benign (μ={bb_dist.mean():.3f})', density=True)
ax.hist(pp_dist, bins=bins, alpha=0.5, color='#e74c3c', label=f'Phish-Phish (μ={pp_dist.mean():.3f})', density=True)
ax.hist(bp_dist, bins=bins, alpha=0.5, color='#3498db', label=f'Benign-Phish (μ={bp_dist.mean():.3f})', density=True)
ax.set_xlabel('Euclidean distance in embedding space')
ax.set_ylabel('Density')
ax.set_title('Embedding Space Distance Distributions (Trad Val)', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_embedding_distances.png'), dpi=300, bbox_inches='tight')
plt.show()

# Metric: separation ratio
intra = (bb_dist.mean() + pp_dist.mean()) / 2
inter = bp_dist.mean()
ratio = inter / intra if intra > 0 else 0
print(f'\nIntra-class avg distance:  {intra:.4f}')
print(f'Inter-class avg distance:  {inter:.4f}')
print(f'Separation ratio (>1 = good): {ratio:.4f}')

---
## 5. SHAP ANALYSIS on embedding features

In [ ]:
# ============================================================
# 5.1 SHAP on the classifier HEAD (embeddings → logit)
# ============================================================
# We treat the 128-d Siamese embeddings as "features" and use SHAP to see
# which dimensions of the embedding contribute most to the phishing decision.
import shap

# Use a subset for speed
n_explain = min(500, len(embeddings))
idx_sample = np.random.choice(len(embeddings), n_explain, replace=False)
X_emb = embeddings[idx_sample]
y_emb = lab_val[idx_sample]

# Wrapper that takes embeddings → sigmoid prob
def predict_from_emb(emb_np):
    with torch.no_grad():
        emb = torch.from_numpy(emb_np.astype(np.float32)).to(device)
        logits = classifier.head(emb)
        return torch.sigmoid(logits).cpu().numpy().flatten()

# Background for SHAP explainer
background = X_emb[np.random.choice(len(X_emb), 50, replace=False)]
explainer = shap.KernelExplainer(predict_from_emb, background)

# Compute SHAP values (slow; use small subset)
X_explain = X_emb[:100]
print('Computing SHAP values (100 samples × 128 features)...')
shap_values = explainer.shap_values(X_explain, nsamples=50)
print(f'SHAP values shape: {np.array(shap_values).shape}')

# Mean absolute SHAP per embedding dim
mean_shap = np.abs(shap_values).mean(axis=0)
top_dims = np.argsort(mean_shap)[::-1][:20]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(20), mean_shap[top_dims][::-1], color='#9b59b6', edgecolor='black')
ax.set_yticks(range(20))
ax.set_yticklabels([f'emb_dim_{d}' for d in top_dims[::-1]])
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Top 20 Embedding Dimensions by SHAP Importance', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_shap_embedding.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_shap_embedding.png')

---
## 6. SUMMARY

In [ ]:
# ============================================================
# 6. FINAL SUMMARY
# ============================================================
print('='*60)
print(' Q-SHIELD v3 XAI ANALYSIS — COMPLETED')
print('='*60)
print(f'\nGrad-CAM sample figures:     fig_gradcam_samples.png')
print(f'Aggregate attention maps:    fig_gradcam_aggregate.png')
print(f'Embedding distance analysis: fig_embedding_distances.png')
print(f'SHAP importance:             fig_shap_embedding.png')
print(f'\nSeparation ratio (val): {ratio:.4f} (>1.0 indicates class separation)')
print(f'\nAll figures saved to {BASE}/')